# 导

In [3]:
import requests,json,os,time,sys,re
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
pool = ThreadPoolExecutor(max_workers=16,)
from datetime import datetime, timedelta
今天 = datetime.now().strftime("%Y-%m-%d")
def 真的今天():
     return time.strftime("%Y-%m-%d",time.localtime(time.time()))
import re
import pandas as pd

from tqdm.auto import tqdm
import urllib3
urllib3.disable_warnings()

#from pyquery import PyQuery as pq

# 开抓！！
## def

In [4]:

def 查研报(查询词,查询方式):
    每页数量=100
    表单={
        查询方式:	查询词,  #title text  attachContent
    'wechatReport':	1,
    'permssionReport'	:1,
    'pageIndex'	:1,
    'limit'	:每页数量,
    'sort':	'datetime',
    'order':	'desc',}
    url='https://choicewzp1.eastmoney.com/Report/Search.do?'
    大结果lt=[]
    #进度条=tqdm(desc='查询词:%s'%(查询词))
    while True:
        response = requests.get(url,params=表单,timeout=500, verify=False)#cookies=cookies,
        response.close()
        if response.ok: 
            pass
        else:
            print(response.reason)
            break
        JSdt1=response.json()
        if 表单['pageIndex']==1:
            进度条=tqdm(desc='%s:%s'%(查询方式,查询词),total=int(JSdt1['total']))
        if int(JSdt1['total'])==0:
            进度条.set_description('完成！')
            break
        结果=pd.DataFrame(JSdt1['records'])
        大结果lt.append(结果)
        表单['pageIndex']+=1
        进度条.update(结果.shape[0])
    大结果= pd.concat(大结果lt)
    return 大结果
def remove_illegal_chars(filename):
    illegal_chars = r'[<>:"//////\\|?*]'
    new_filename = re.sub(illegal_chars, '', filename)
    return new_filename
def 数据处理(大结果):
    大结果['sourceurl']=大结果['sourceurl'].replace('',None)
    大结果['是否网页']=~ 大结果['sourceurl'].isna()
    大结果=大结果.fillna(value={'sourceurl':大结果['attach'].apply(lambda x : x[0]['url']  )})
    大结果['文件名']=大结果['attach'].apply(lambda x : remove_illegal_chars(x[0]['name'] )  )
    保留列=['officialaccount','order',	'org','rtype',	'sortTime'	,'sourceurl','title',	'是否网页'	,'文件名',]
    大结果=大结果[保留列].set_index('order')
    大结果['权限']=None
    return 大结果
def 多线程下载保存(总研报处理,保存位置,查询):
    if not os.path.exists(保存位置):os.makedirs(保存位置)
    def MT下载(行it):
        权限=pd.Series([None],index=[行it[0]] )
        权限.name='权限'
        文件名=os.path.join(保存位置,行it[1]['文件名'] )
        if os.path.exists(文件名):
            权限.iloc[0]=True
            return 权限
        if 行it[1]['权限']==False:return None
        if 行it[1]['是否网页']==False:
            response = requests.get(行it[1]['sourceurl'],verify=False)
            response.close
            if response.status_code == 200:
                with open(文件名, 'wb') as f:
                    f.write(response.content)
                权限.iloc[0]=True
            else:
                权限.iloc[0]=False
        return 权限
    re1=list(tqdm(pool.map(MT下载,总研报处理.iterrows()),desc='下载研报',total=总研报处理.shape[0]))
    权限1=pd.concat(re1)
    总研报处理=总研报处理.fillna({'权限':权限1})
    print('有效下载',总研报处理['权限'].sum())
    总研报处理.to_csv('%s.csv'%(保存位置))
    return 总研报处理

## 开抓

下载研报:   0%|          | 0/1873 [00:00<?, ?it/s]

In [ ]:
前置位置='/home/tongjisem/Mzb/CH爬虫数据/研报'
查询词='城投债'
查询='按标题'
查询方式={'按标题':'title' ,'按摘要':'text' ,'按附件': 'attachContent'}
总研报=查研报(查询词,查询方式[查询])
总研报处理=数据处理(总研报)
保存位置=os.path.join(前置位置,查询+'-'+查询词)
总研报处理=多线程下载保存(总研报处理,保存位置,查询)